###SQL ASSIGNMENT
ANALYSING THE SIZWE COFFEE BUSINESS

### **Task 1 - Channel performance**
Nomsa needs the revenue split by channel for FY2024 and FY2025 side by side, with the growth rate
between them. One row per channel, three figures on each.
FY2024 is orders placed on or before 30 June 2024. FY2025 is everything after that.

In [0]:
%sql
WITH channel_fy AS (
    SELECT so.channel,
           SUM(CASE WHEN so.order_date <= '2024-06-30' THEN sl.line_revenue ELSE 0 END) AS fy2024_revenue,
           SUM(CASE WHEN so.order_date >  '2024-06-30' THEN sl.line_revenue ELSE 0 END) AS fy2025_revenue
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    GROUP BY so.channel
)
SELECT channel,
       ROUND(fy2024_revenue, 2) AS fy2024_revenue,
       ROUND(fy2025_revenue, 2) AS fy2025_revenue,
       ROUND(100.0 * (fy2025_revenue - fy2024_revenue) / NULLIF(fy2024_revenue, 0), 1) AS growth_pct
FROM channel_fy
ORDER BY fy2025_revenue DESC

### Task 2 - Where the profit actually comes from
For each product line, return revenue, cost of goods, gross profit and gross margin percentage. Sort by
gross profit. In one or two sentences under your query: which line earns the most profit, and is it the same line that
earns the highest margin?

In [0]:
%sql
SELECT p.product_line,
       ROUND(SUM(sl.line_revenue), 2) AS revenue,
       ROUND(SUM(sl.cogs), 2) AS cost_of_goods,
       ROUND(SUM(sl.line_revenue) - SUM(sl.cogs), 2) AS gross_profit,
       ROUND(100.0 * (SUM(sl.line_revenue) - SUM(sl.cogs))
             / NULLIF(SUM(sl.line_revenue), 0), 2) AS gross_margin_pct
FROM sizwe_sales_lines AS sl
INNER JOIN sizwe_products AS p
    ON sl.sku = p.sku
GROUP BY p.product_line
ORDER BY gross_profit DESC

--House Blend has the highest revenue, however Single Origin has the highest Gross Margin Percentage. It is not the same line.

### Task 3 - The collections problem
Nomsa suspects the business is being paid more slowly than it used to be, and wants to know where the
problem sits. For each channel, return the number of invoices, how many have been settled, how many are still open,
the average number of days taken to settle, and the total value still outstanding.
In two or three sentences: what does this tell you, and which channel would you investigate first?

In [0]:
%sql
SELECT c.channel,
       COUNT(*) AS invoices,
       COUNT(ar.settlement_date) AS settled_invoices,
       SUM(CASE WHEN ar.settlement_date IS NULL THEN 1 ELSE 0 END) AS open_invoices,
       ROUND(AVG(DATEDIFF(DAY, ar.invoice_date, ar.settlement_date)), 1) AS avg_days_to_settle,
       ROUND(SUM(ar.invoice_amount - COALESCE(ar.amount_settled, 0)), 2) AS outstanding_value
FROM sizwe_ar_ledger AS ar
INNER JOIN sizwe_customers AS c
    ON ar.customer_id = c.customer_id
GROUP BY c.channel
ORDER BY outstanding_value DESC

--Nomsa is right. I would investigate the Wholesale channel first. It has 3 454 open invoices which amount to R87.7m. The average days to settle are high. With closing these invoices it would increase the average for the current financial year.

### Task 4 - The ten accounts that matter most
Bongani wants a one-page list of the ten largest customers by FY2025 revenue, showing the trading
name, segment, province, the sales representative who owns the account, and the revenue.
Not every customer has a sales representative. Make sure your query does not silently drop anyone.

In [0]:
%sql
WITH fy2025_rev AS (
    SELECT so.customer_id,
           SUM(sl.line_revenue) AS revenue
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    WHERE so.order_date > '2024-06-30'
    GROUP BY so.customer_id
)
SELECT c.trading_name,
       c.segment,
       c.province,
       COALESCE(r.rep_name, 'Unassigned') AS sales_rep,
       ROUND(fr.revenue, 2) AS fy2025_revenue
FROM fy2025_rev AS fr
INNER JOIN sizwe_customers AS c
    ON fr.customer_id = c.customer_id
LEFT JOIN sizwe_reps AS r
    ON c.rep_id = r.rep_id
ORDER BY fr.revenue DESC
LIMIT 10

### Task 5 - A commercial gap
Trade promotions are how Sizwe supports its wholesale accounts. Bongani thinks some sizeable
customers have been getting nothing.
Return the fifteen highest-revenue Wholesale customers who have never received a trade promotion,
with their segment and province.
One sentence: what would you recommend doing about the top three?

In [0]:
%sql
WITH customer_rev AS (
    SELECT so.customer_id,
           SUM(sl.line_revenue) AS revenue
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    GROUP BY so.customer_id
)
SELECT c.customer_id,
       c.trading_name,
       c.segment,
       c.province,
       ROUND(cr.revenue, 2) AS revenue
FROM sizwe_customers AS c
INNER JOIN customer_rev AS cr
    ON c.customer_id = cr.customer_id
WHERE c.channel = 'Wholesale'
  AND NOT EXISTS (SELECT 1
                  FROM sizwe_trade_promotions AS tp
                  WHERE tp.customer_id = c.customer_id)
ORDER BY cr.revenue DESC
LIMIT 15

--Bongani is right. Umhlanga Fresh 108, Ridgeway Foods 308 and Kalahari Kitchen 42 are among the six biggest customers in the whole business, yet they’ve never had any promotional support. I'd recommend the reps to setup trade promotion support immediately. Customers that spend a large amount of money and settle their accounts need a reward so they can be retained.

### Task 6 - Are the sales representatives hitting target?
Each sales representative carries a monthly revenue target. Return every sales representative with their
territory, their monthly target, their actual FY2025 monthly average, and that average as a percentage of
target. Sort by percentage of target, best first.
Every sales representative must appear, including any who generated no revenue.

In [0]:
%sql
WITH rep_fy2025 AS (
    SELECT so.rep_id,
           SUM(sl.line_revenue) AS revenue
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    WHERE so.order_date > '2024-06-30'
    GROUP BY so.rep_id
)
SELECT r.rep_id,
       r.rep_name,
       r.territory,
       r.target_monthly,
       ROUND(COALESCE(rf.revenue, 0) / 12, 2) AS fy2025_monthly_avg,
       ROUND(100.0 * COALESCE(rf.revenue, 0) / 12
             / NULLIF(r.target_monthly, 0), 2) AS pct_of_target
FROM sizwe_reps AS r
LEFT JOIN rep_fy2025 AS rf
    ON r.rep_id = rf.rep_id
ORDER BY pct_of_target DESC

###Task 7 - The shape of the year
Return monthly revenue across the full two years, with the percentage change on the previous month
and a rolling three-month average alongside it.
In two sentences: is there a seasonal pattern, and when is Sizwe's quietest month?

In [0]:
%sql
WITH monthly_rev AS (
    SELECT DATE_TRUNC('month', so.order_date) AS month,
           SUM(sl.line_revenue) AS revenue
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    GROUP BY 1
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / NULLIF(LAG(revenue) OVER (ORDER BY month), 0), 1) AS pct_change_vs_prev_month,
       ROUND(AVG(revenue) OVER (ORDER BY month
                                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS rolling_3m_avg
FROM monthly_rev
ORDER BY month

--Yes, there is a clear seasonal pattern, and it repeats both years. Revenue is highest around June/July, falls every month from August to December, then climbs steadily from January back up to June. December is the quietest month, both times, about 13% down on November. The growth is 8.8% on both financial years shown in Task 1.

###Task 8 - Why has the margin moved?
Gross margin fell from 34.7% in FY2024 to 33.0% in FY2025, while revenue grew nearly 9%. Nobody
can explain it.
For each product line, return revenue and gross margin for each financial year, plus that line's share of
total revenue in that year.
Then answer, in a short paragraph: what caused the margin decline? Your answer must point at specific
numbers in your result.

In [0]:
%sql
WITH line_fy AS (
    SELECT p.product_line,
           CASE WHEN so.order_date <= '2024-06-30' THEN 'FY2024' ELSE 'FY2025' END AS fin_year,
           SUM(sl.line_revenue) AS revenue,
           SUM(sl.cogs) AS cogs
    FROM sizwe_sales_orders AS so
    INNER JOIN sizwe_sales_lines AS sl
        ON so.order_id = sl.order_id
    INNER JOIN sizwe_products AS p
        ON sl.sku = p.sku
    GROUP BY 1, 2
)
SELECT product_line,
       fin_year,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - cogs) / NULLIF(revenue, 0), 1) AS gross_margin_pct,
       ROUND(100.0 * revenue / SUM(revenue) OVER (PARTITION BY fin_year), 1) AS revenue_share_pct
FROM line_fy
ORDER BY product_line, fin_year

--The cause for the margin decline is sales mix, not margin erosion within lines. The Bulk Commercial’s revenue share rose from 25.0% to 33.6%, and steady 19%.9% gross margin in both years. Capsules show a drop. Decaf and House Blend gross margin stayed the same at 35.8% but a drop in revenue share. With Single Origin having a 0.1% growth but slight decline in revenue share.